In [1]:
import pandas as pd
import geopandas as gpd
import altair as alt

from shapely.geometry import Point

In [13]:
df = pd.read_csv('../../data/Taxi_Trips.csv')
geometry = [Point(xy) for xy in zip(df['Pickup Centroid Longitude'], df['Pickup Centroid Latitude'])]
gdf = gpd.GeoDataFrame(df, geometry=geometry, crs=4326) 

In [4]:
alt.Chart(gdf.sample(500)).mark_geoshape()

alt.Chart(...)

In [5]:
alt.Chart(gdf.sample(500)).mark_geoshape().encode(color='Fare')

alt.Chart(...)

In [14]:
# Choropleth (value based)
chicago = gpd.read_file('../../data/chicago.geojson')

gdf = gpd.sjoin(gdf, chicago, predicate='within')
joined = gdf.groupby('zip').agg({'Fare': 'mean'})
joined = joined.filter(['Fare'])

merged = chicago.merge(joined, on='zip')

alt.Chart(merged).mark_geoshape().encode(color='Fare').project(type='mercator')

alt.Chart(...)

In [15]:
choropleth = alt.Chart(merged).mark_geoshape().encode(color='Fare').project(type='mercator')

bar = alt.Chart(merged.nlargest(15, "Fare"), title="Top 15 ZIP codes by fare").mark_bar().encode(
        x="Fare",
        y=alt.Y("zip").sort("-x"),
    )

In [16]:
click_zip = alt.selection_point(fields=["zip"])
opacity = alt.when(click_zip).then(alt.value(1)).otherwise(alt.value(0.2))

choropleth = alt.Chart(merged).mark_geoshape().encode(color='Fare').project(type='mercator').encode(opacity=opacity)

(choropleth).add_params(click_zip)

alt.Chart(...)

In [17]:
click_zip = alt.selection_point(fields=["zip"])
opacity = alt.when(click_zip).then(alt.value(1)).otherwise(alt.value(0.2))

choropleth = alt.Chart(merged).mark_geoshape().encode(color='Fare').project(type='mercator').encode(opacity=opacity)

bar = alt.Chart(merged.nlargest(15, "Fare"), title="Top 15 ZIP codes by fare").mark_bar().encode(
        x="Fare",
        opacity=opacity,
        y=alt.Y("zip").sort("-x"),
    )

(choropleth & bar).add_params(click_zip)

alt.VConcatChart(...)